# 03 · Points scorecard (weight of evidence)

How collections scores are usually delivered: each feature is cut into bins, each bin gets a weight of evidence
(WoE = log of payers' share ÷ non-payers' share), features with too little information value (IV) are dropped, and a
logistic regression turns WoE into points an agent can read.

| IV | Strength (credit-scoring convention) |
|---|---|
| < 0.02 | useless |
| 0.02 – 0.1 | weak |
| 0.1 – 0.3 | medium |
| > 0.3 | strong |

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "03"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
X, y = M.xy(dev, M.FS_ALL)
iv = pd.Series(M.WoE().fit(X, y).iv_).sort_values(ascending=False)
plt.figure(figsize=(7, 0.3 * len(iv) + 1)); iv.plot.barh(color="#4C72B0"); plt.axvline(.02, ls="--", c="grey")
plt.gca().invert_yaxis(); plt.title("Information value per feature (all training rows)"); plt.show()

In [ ]:
run("SC-01", M.pipeline(M.preprocess("woe"), M.logreg(C=1.0)), "FS_ALL", "WoE (6 quantile bins) + logistic, all features")
run("SC-02", M.pipeline(M.preprocess("woe"), M.IVFilter(.02), M.logreg(C=1.0)), "FS_ALL", "WoE + IV filter (>0.02) + logistic")
run("SC-03", M.pipeline(M.preprocess("woe"), M.IVFilter(.02), M.CorrelationPruner(.7), M.logreg(C=1.0)), "FS_ALL", "WoE + IV filter + correlation pruning (|rho|>0.7)")
run("SC-04", M.pipeline(M.preprocess("woe_mono"), M.IVFilter(.02), M.logreg(C=1.0)), "FS_ALL", "Monotonic WoE + IV filter + logistic")
run("SC-05", M.pipeline(M.preprocess("woe_mono"), M.IVFilter(.02), M.logreg(C=1.0)), "FS_TRANSFER", "Monotonic WoE + IV filter, FS_TRANSFER (the deliverable scorecard)");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

## The scorecard (SC-05 refitted on all training rows)
Points = 600 at even odds, +20 points = double the odds of paying. A patient's score is the sum of their bins' points.

In [ ]:
X, y = M.xy(dev, M.FS_TRANSFER)
sc = M.pipeline(M.preprocess("woe_mono"), M.IVFilter(.02), M.logreg(C=1.0)).fit(X, y)
woe, ivf, lr = sc[0], sc[1], sc[-1]
factor = 20 / np.log(2); n = len(ivf.keep_)
rows = []
for f, beta in zip(ivf.keep_, lr.coef_[0]):
    edges = woe.edges_[f]
    for b, w in sorted(woe.table_[f].items()):
        if b == -1: label = "missing"
        else:
            lo = -np.inf if b == 0 else edges[int(b) - 1]; hi = np.inf if b == len(edges) else edges[int(b)]
            label = f"[{lo:,.0f}, {hi:,.0f})"
        rows.append({"feature": f, "bin": label, "WoE": w, "points": factor * (beta * w + lr.intercept_[0] / n)})
card = pd.DataFrame(rows); card["points"] = card.points.round(0).astype(int) + round(600 / n)
card

## Findings (Salud-tuned world)
- All WoE variants score 0.870–0.872, level with tuned logistic regression. Monotone bins (SC-04) cost nothing.
- **SC-05**, the deliverable scorecard on FS_TRANSFER, reaches 0.871 with about 10 features and a calibration slope of 0.99, the best-calibrated model in this notebook.
- Correlation pruning at |ρ| > 0.7 (SC-03) costs 0.014 (p = 0.06). Pruning that aggressively removes real signal.